In [10]:
import pandas as pd 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [11]:
qb_df = pd.read_csv('/Users/christianmarchello/Documents/howamachinelearns/qb_vs_def_ytd_norm.csv')
rb_df = pd.read_csv('/Users/christianmarchello/Documents/howamachinelearns/rb_vs_def_ytd_norm.csv')
wr_df = pd.read_csv('/Users/christianmarchello/Documents/howamachinelearns/wr_vs_def_ytd_norm.csv')
te_df = pd.read_csv('/Users/christianmarchello/Documents/howamachinelearns/te_vs_def_ytd_norm.csv')
k_df = pd.read_csv('/Users/christianmarchello/Documents/howamachinelearns/k_vs_def_ytd_norm.csv')
dst_df = pd.read_csv('/Users/christianmarchello/Documents/howamachinelearns/dst_vs_def_ytd_norm.csv')


In [22]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ------------------------------
# 1) Which columns “tell the story” for props?
# ------------------------------
cfg = {
    'QB': {  # passing + rushing
        'df':  qb_df,
        'features': ['Pass_Yds','Pass_TD','Pass_Int','Rush_Yds','Rush_Att','Rush_TD'],
        # flags you want to target as props
        'flag_defs': {
            'QB_PASSY_hot':   ['Pass_Yds'],                        # passing yards
            'QB_PTD_hot':     ['Pass_TD'],                        # passing TDs
            'QB_RUSH_hot':    ['Rush_Yds','Rush_Att'],        # QB rushing
        },
    },
    'RB': {  # rushing + receiving volume/yards
        'df':  rb_df,
        'features': ['Rush_Yds','Rush_Avg','Rush_TD','Tgt','Rec','Rec_Yds','Rec_Avg','Rec_TD'],
        'flag_defs': {
            'RB_RUSH_hot':    ['Rush_Yds','Rush_Att' if 'Rush_Att' in rb_df.columns else 'Rush_Yds'],
            'RB_REC_hot':     ['Tgt','Rec'],                 # receptions/targets
            'RB_RECY_hot':    ['Rec_Yds'],                   # receiving yards
        },
    },
    'WR': {  # receiving volume/yards
        'df':  wr_df,
        'features': ['Tgt','Rec','Rec_Yds','Rec_Avg','Rec_TD'],
        'flag_defs': {
            'WR_REC_hot':     ['Tgt','Rec'],
            'WR_RECY_hot':    ['Rec_Yds'],
        },
    },
    'TE': {  # receiving volume/yards
        'df':  te_df,
        'features': ['Tgt','Rec','Rec_Yds','Rec_Avg','Rec_TD'],
        'flag_defs': {
            'TE_REC_hot':     ['Tgt','Rec'],
            'TE_RECY_hot':    ['Rec_Yds'],
            'TE_TD_hot':      ['Rec_TD'],
        },
    },
    'K': {   # attempt/volume profile
        'df':  k_df,
        'features': ['FGA','FG','XP','XPAtt','FPTS'],
        'flag_defs': {
            'K_ATTEMPT_hot':  ['FGA','XPAtt'],              # lots of attempts
            'K_FPTS_hot':     ['FPTS'],
        },
    }
}

# Percentile threshold for “hotspot”
PCT = 0.75  # top quartile; tighten or loosen as you like

def _percentile_flags(df, team_col, cols, q=PCT):
    """Return a boolean flag True if any of the columns is >= league q-quantile."""
    flags = {}
    for c in cols:
        cut = df[c].quantile(q)
        flags[c] = (df[c] >= cut)
    any_hot = pd.concat(flags, axis=1).any(axis=1)
    return any_hot

def _cluster_position(df, team_col='Team', use_cols=None, drop_cols=('Rank',), k_range=range(3,8)):
    """Cluster a single position on selected numeric columns."""
    X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    if use_cols:
        X = X[[c for c in use_cols if c in X.columns]]
    X = X.select_dtypes('number').copy()
    sc = StandardScaler().fit(X)
    Z  = sc.transform(X)
    best = (-1, None, None)  # (silhouette, labels, model)
    for k in k_range:
        km = KMeans(n_clusters=k, n_init='auto', random_state=42)
        lbl = km.fit_predict(Z)
        s   = silhouette_score(Z, lbl)
        if s > best[0]:
            best = (s, lbl, km)
    labels, model = best[1], best[2]
    dist_min = model.transform(Z).min(axis=1)
    return labels, dist_min, model, sc, X.columns.tolist()

# ------------------------------
# 2) Build flags + clusters per position
# ------------------------------
frames = []
for pos, spec in cfg.items():
    df = spec['df'].copy()
    df_num = df.select_dtypes('number')
    # Sanity: keep only teams in this CSV
    teams = df['Team'].values

    # (a) Team-level flags (league percentile)
    per_team_flags = {}
    for flag_name, cols in spec['flag_defs'].items():
        cols = [c for c in cols if c in df.columns]
        if not cols: 
            continue
        per_team_flags[flag_name] = _percentile_flags(df, 'Team', cols, q=PCT).values

    # (b) Per-position clustering on the chosen features
    labels, dmin, model, scaler, used_cols = _cluster_position(df, 'Team', spec['features'])

    # (c) Cluster-level hotspot labels (cluster means vs league 75th pct) for the *same* columns you care about
    cluster_flags = {}
    tmp = df[['Team'] + list(set(sum(spec['flag_defs'].values(), [])) if spec['flag_defs'] else [])].copy()
    tmp['cluster'] = labels
    for flag_name, cols in spec['flag_defs'].items():
        cols = [c for c in cols if c in tmp.columns]
        if not cols: continue
        # compute cluster mean of the "flag driver" columns using a simple average of selected cols
        tmp['_flag_driver'] = tmp[cols].mean(axis=1)
        league_cut = tmp['_flag_driver'].quantile(PCT)
        cmeans = tmp.groupby('cluster')['_flag_driver'].mean()
        hot_clusters = set(cmeans[cmeans >= league_cut].index.tolist())
        cluster_flags[flag_name + '_cluster'] = np.isin(labels, list(hot_clusters))

    # (d) Assemble this position’s frame
    pos_df = pd.DataFrame({'Team': teams, f'{pos}_cluster': labels, f'{pos}_dist': dmin})
    # add team-level flags
    for k, v in per_team_flags.items():
        pos_df[f'{pos}_{k}'] = v
    # add cluster-level flags
    for k, v in cluster_flags.items():
        pos_df[f'{pos}_{k}'] = v

    frames.append(pos_df)

# ------------------------------
# 3) Merge all positions into one prop-ready table
# ------------------------------
prop_flags = frames[0]
for f in frames[1:]:
    prop_flags = prop_flags.merge(f, on='Team', how='inner')

prop_flags.sort_values('Team').head(32)


,Team,QB_cluster,QB_dist,QB_QB_PASSY_hot,QB_QB_PTD_hot,QB_QB_RUSH_hot,QB_QB_PASSY_hot_cluster,QB_QB_PTD_hot_cluster,QB_QB_RUSH_hot_cluster,RB_cluster,...,TE_TE_TD_hot,TE_TE_REC_hot_cluster,TE_TE_RECY_hot_cluster,TE_TE_TD_hot_cluster,K_cluster,K_dist,K_K_ATTEMPT_hot,K_K_FPTS_hot,K_K_ATTEMPT_hot_cluster,K_K_FPTS_hot_cluster
18,49ers,0,1.822055,False,False,False,False,False,False,0,...,True,True,False,False,1,1.161441,False,False,False,False
6,Bears,0,3.211023,False,True,False,False,False,False,2,...,False,True,False,False,6,0.592431,False,False,False,False
8,Bengals,0,2.252352,True,True,False,False,False,False,2,...,True,True,False,False,3,0.000000,True,False,True,False
20,Bills,1,1.564338,False,False,True,False,False,True,1,...,False,False,False,False,1,0.817394,False,False,False,False
30,Broncos,0,2.355642,False,False,False,False,False,False,0,...,False,False,False,False,0,0.909825,True,True,False,True
25,Browns,0,1.898385,False,False,False,False,False,False,1,...,True,False,False,False,6,0.868279,True,False,False,False
4,Buccaneers,2,1.392789,False,True,True,True,True,True,0,...,True,True,False,False,6,0.255486,True,False,False,False
29,Cardinals,0,1.852975,True,False,False,False,False,False,0,...,False,False,True,False,0,0.475928,True,True,False,True
23,Chargers,1,1.501264,False,False,True,False,False,True,1,...,False,False,False,False,0,0.712001,False,True,False,True
22,Chiefs,1,1.963415,False,False,True,False,False,True,0,...,False,False,False,False,1,0.746505,False,False,False,False


In [24]:
prop_flags.to_csv('prop_flags.csv', index=False)


In [27]:
import pandas as pd
import numpy as np

# ---- scoring helpers ----
def score_pair(team_hot: bool, clus_hot: bool) -> int:
    return (2 if team_hot else 0) + (1 if clus_hot else 0)

def add_dist_penalty(raw_score: int, dist: float, hi=1.6, lo=1.0) -> float:
    # light penalty for being far from centroid; tiny bonus if very prototypical
    s = float(raw_score)
    if dist > hi:  s -= 0.5
    if dist < lo:  s += 0.2
    return s

def label_from_score(s: float) -> str:
    if s >= 2.5:   return "STRONG"
    if s >= 1.0:   return "MILD"
    return "AVOID"

# ---- which flags belong to each stat ----
CFG = {
    "RB": {
        "dist": "RB_dist",
        "stats": {
            "RB_RUSH": ("RB_RB_RUSH_hot", "RB_RB_RUSH_hot_cluster"),
            "RB_REC" : ("RB_RB_REC_hot",  "RB_RB_REC_hot_cluster"),
            "RB_RECY": ("RB_RB_RECY_hot", "RB_RB_RECY_hot_cluster"),
        },
    },
    "QB": {
        "dist": "QB_dist",
        "stats": {
            "QB_PASSY": ("QB_QB_PASSY_hot", "QB_QB_PASSY_hot_cluster"),
            "QB_PTD"  : ("QB_QB_PTD_hot",   "QB_QB_PTD_hot_cluster"),
            "QB_RUSH" : ("QB_QB_RUSH_hot",  "QB_QB_RUSH_hot_cluster"),
        },
    },
    "WR": {
        "dist": "WR_dist",
        "stats": {
            "WR_REC" : ("WR_WR_REC_hot",  "WR_WR_REC_hot_cluster"),
            "WR_RECY": ("WR_WR_RECY_hot", "WR_WR_RECY_hot_cluster"),
        },
    },
    "TE": {
        "dist": "TE_dist",
        "stats": {
            "TE_REC" : ("TE_TE_REC_hot",  "TE_TE_REC_hot_cluster"),
            "TE_RECY": ("TE_TE_RECY_hot", "TE_TE_RECY_hot_cluster"),
            "TE_TD"  : ("TE_TE_TD_hot",   "TE_TE_TD_hot_cluster"),
        },
    },
    "K": {
        "dist": "K_dist",
        "stats": {
            "K_ATTEMPT": ("K_K_ATTEMPT_hot", "K_K_ATTEMPT_hot_cluster"),
            "K_FPTS"   : ("K_K_FPTS_hot",    "K_K_FPTS_hot_cluster"),
        },
    },
}

def best_pick_for_pos(row: pd.Series, pos: str):
    """Return (label, best_stat, score, reason) for a single position on this row."""
    dist_col = CFG[pos]["dist"]
    dist = row.get(dist_col, 1.3)  # default mid if not present

    best = ("AVOID", None, -1.0, "")
    for stat, (th, ch) in CFG[pos]["stats"].items():
        team_hot   = bool(row.get(th, False))
        clust_hot  = bool(row.get(ch, False))
        raw = score_pair(team_hot, clust_hot)
        s = add_dist_penalty(raw, dist)
        lab = label_from_score(s)

        # keep the highest score; tie-breaker prefers team_hot True
        tie = (abs(s - best[2]) < 1e-9) and team_hot and (not (" team_hot" in best[3]))
        if (s > best[2]) or tie:
            reason = f"{stat}: team_hot={team_hot}, cluster_hot={clust_hot}, dist={dist:.2f}, score={s:.2f}"
            best = (lab, stat, s, reason)

    return best  # (label, stat, score, reason)

def condense_picks(prop_flags: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in prop_flags.iterrows():
        out = {"Team": r["Team"]}
        overall = ("AVOID", None, -1.0, "", "")

        for pos in ["RB","QB","WR","TE","K"]:
            lab, stat, sc, why = best_pick_for_pos(r, pos)
            out[f"{pos}_pick"]   = lab
            out[f"{pos}_stat"]   = stat
            out[f"{pos}_score"]  = round(sc, 2)
            out[f"{pos}_reason"] = why

            # update overall
            if sc > overall[2]:
                overall = (lab, stat, sc, pos, why)

        out["OVERALL_pick"]   = overall[0]
        out["OVERALL_area"]   = overall[3]      # which position drove it
        out["OVERALL_stat"]   = overall[1]
        out["OVERALL_score"]  = round(overall[2], 2)
        out["OVERALL_reason"] = overall[4]
        rows.append(out)

    cols_order = ["Team",
                  "RB_pick","RB_stat","RB_score",
                  "QB_pick","QB_stat","QB_score",
                  "WR_pick","WR_stat","WR_score",
                  "TE_pick","TE_stat","TE_score",
                  "K_pick","K_stat","K_score",
                  "OVERALL_pick","OVERALL_area","OVERALL_stat","OVERALL_score",
                  "RB_reason","QB_reason","WR_reason","TE_reason","K_reason","OVERALL_reason"]
    df = pd.DataFrame(rows)
    # reorder if all columns exist
    return df[[c for c in cols_order if c in df.columns]]

# ---- run it ----
prop_picks = condense_picks(prop_flags)

# peek
prop_picks.head(10)

prop_picks.to_csv('prop_picks_summary.csv', index=False)


In [30]:
import pandas as pd

# Which columns hold the cluster + distance for each position
POS = {
    "QB": ("QB_cluster", "QB_dist"),
    "RB": ("RB_cluster", "RB_dist"),
    "WR": ("WR_cluster", "WR_dist"),
    "TE": ("TE_cluster", "TE_dist"),
    "K" : ("K_cluster",  "K_dist"),
}

cluster_tables = {}

for pos, (ccol, dcol) in POS.items():
    df = prop_flags[['Team', ccol, dcol]].copy()
    df = df.sort_values([ccol, dcol], ascending=[True, True])

    # rank distance inside cluster so we can identify prototypes
    df['rank_in_cluster'] = df.groupby(ccol)[dcol].rank(method='first')

    # table: cluster → size, 3 closest examples, all teams (comma separated)
    tab = (
        df.groupby(ccol)
          .agg(
              n_teams=('Team', 'count'),
              prototypes=('Team', lambda s: ', '.join(s.iloc[:3])),  # 3 closest
              teams=('Team', lambda s: ', '.join(list(s)))
          )
          .reset_index()
          .rename(columns={ccol: f'{pos}_cluster'})
          .sort_values('n_teams', ascending=False)
          .reset_index(drop=True)
    )

    cluster_tables[pos] = tab
    print(f"\n=== {pos} clusters ===")
    display(tab)

    # optional: save
    # tab.to_csv(f'{pos.lower()}_clusters_roster.csv', index=False)

# (Optional) one long table for all positions
long = []
for pos, (ccol, dcol) in POS.items():
    tmp = prop_flags[['Team', ccol, dcol]].copy()
    tmp = tmp.rename(columns={ccol:'cluster', dcol:'dist'})
    tmp['position'] = pos
    long.append(tmp)

all_pos_long = pd.concat(long, ignore_index=True)
# examples:
display(all_pos_long.sort_values(['position','cluster','dist']))




=== QB clusters ===


,QB_cluster,n_teams,prototypes,teams
0,0,21,"Panthers, Rams, Patriots","Panthers, Rams, Patriots, Colts, Titans, Raide..."
1,2,6,"Buccaneers, Saints, Ravens","Buccaneers, Saints, Ravens, Cowboys, Steelers,..."
2,1,5,"Eagles, Chargers, Bills","Eagles, Chargers, Bills, Chiefs, Falcons"



=== RB clusters ===


,RB_cluster,n_teams,prototypes,teams
0,0,17,"Cardinals, Steelers, Saints","Cardinals, Steelers, Saints, Chiefs, Panthers,..."
1,1,8,"Texans, Chargers, Raiders","Texans, Chargers, Raiders, Lions, Browns, Bill..."
2,2,7,"Eagles, Ravens, Giants","Eagles, Ravens, Giants, Titans, Bengals, Dolph..."



=== WR clusters ===


...


=== TE clusters ===


,TE_cluster,n_teams,prototypes,teams
0,2,13,"Vikings, 49ers, Rams","Vikings, 49ers, Rams, Seahawks, Giants, Jaguar..."
1,1,12,"Chargers, Texans, Eagles","Chargers, Texans, Eagles, Chiefs, Broncos, Tit..."
2,0,7,"Patriots, Dolphins, Panthers","Patriots, Dolphins, Panthers, Steelers, Ravens..."



=== K clusters ===


,K_cluster,n_teams,prototypes,teams
0,1,10,"Steelers, Patriots, Chiefs","Steelers, Patriots, Chiefs, Packers, Bills, Co..."
1,6,7,"Buccaneers, Saints, Panthers","Buccaneers, Saints, Panthers, Titans, Bears, C..."
2,0,6,"Falcons, Cardinals, Rams","Falcons, Cardinals, Rams, Chargers, Seahawks, ..."
3,5,5,"Eagles, Raiders, Giants","Eagles, Raiders, Giants, Dolphins, Vikings"
4,2,2,"Jets, Ravens","Jets, Ravens"
5,3,1,Bengals,Bengals
6,4,1,Texans,Texans


,Team,cluster,dist,position
155,Falcons,0,0.436618,K
157,Cardinals,0,0.475928,K
147,Rams,0,0.565733,K
151,Chargers,0,0.712001,K
142,Seahawks,0,0.757230,K
...,...,...,...,...
90,Titans,2,1.322091,WR
76,Jaguars,2,1.386032,WR
93,Cardinals,2,1.855330,WR
65,Ravens,2,2.426891,WR


In [23]:
import pandas as pd

# ---------- Helper to summarize one position ----------
def summarize_pos_clusters(
    pos: str,
    prop_flags: pd.DataFrame,
    pos_df: pd.DataFrame,
    cluster_col_map: dict,
    flag_cols_map: dict,
    metrics_map: dict,
):
    """
    pos: 'QB' | 'RB' | 'WR' | 'TE' | 'K'
    prop_flags: the merged table you built (has Team, {pos}_cluster, {pos}_dist, and flags)
    pos_df: the original per-position CSV df for this position (e.g., qb_df for 'QB')
    cluster_col_map: e.g. {'QB':'QB_cluster', 'RB':'RB_cluster', ...}
    flag_cols_map: dict of lists of flag columns in prop_flags for this pos
                   e.g. {'QB':['QB_QB_PASSY_hot','QB_QB_PTD_hot','QB_QB_RUSH_hot',
                               'QB_QB_PASSY_hot_cluster','QB_QB_PTD_hot_cluster','QB_QB_RUSH_hot_cluster']}
    metrics_map: dict of metric columns to average from pos_df for this pos
                 e.g. {'QB':['FPTS','Yd','TD','Int','Rush_Yd']}
    """
    ccol = cluster_col_map[pos]
    # start with cluster + distance + flags
    cols = ['Team', ccol, f'{pos}_dist'] + flag_cols_map[pos]
    base = prop_flags[cols].copy()

    # pull in a few useful raw metrics from the original df
    use_metrics = [c for c in metrics_map.get(pos, []) if c in pos_df.columns]
    met = pos_df[['Team'] + use_metrics].copy()

    g = base.merge(met, on='Team', how='left')

    # aggregate by cluster
    agg_dict = {f'{pos}_dist':'mean'}
    # flags: mean of booleans = fraction True
    for fc in flag_cols_map[pos]:
        agg_dict[fc] = 'mean'
    for m in use_metrics:
        agg_dict[m] = 'mean'

    out = (
        g.groupby(ccol)
         .agg(agg_dict)
         .rename(columns={f'{pos}_dist':'dist_mean'})
         .reset_index()
    )

    # also report cluster size
    counts = g.groupby(ccol)['Team'].nunique().rename('n_teams').reset_index()
    out = counts.merge(out, on=ccol)

    # nicer sorting: more prototypical clusters first (low distance), or by a key flag
    return out.sort_values(['n_teams','dist_mean'], ascending=[False, True]).reset_index(drop=True)

# ---------- Specify which columns to use per position ----------
cluster_col_map = {
    'QB':'QB_cluster', 'RB':'RB_cluster', 'WR':'WR_cluster', 'TE':'TE_cluster', 'K':'K_cluster'
}

flag_cols_map = {
    'QB': [
        'QB_QB_PASSY_hot', 'QB_QB_PTD_hot', 'QB_QB_RUSH_hot',
        'QB_QB_PASSY_hot_cluster', 'QB_QB_PTD_hot_cluster', 'QB_QB_RUSH_hot_cluster'
    ],
    'RB': [
        'RB_RB_RUSH_hot', 'RB_RB_REC_hot', 'RB_RB_RECY_hot',
        'RB_RB_RUSH_hot_cluster', 'RB_RB_REC_hot_cluster', 'RB_RB_RECY_hot_cluster'
    ],
    'WR': [
        'WR_WR_REC_hot', 'WR_WR_RECY_hot',
        'WR_WR_REC_hot_cluster', 'WR_WR_RECY_hot_cluster'
    ],
    'TE': [
        'TE_TE_REC_hot', 'TE_TE_RECY_hot', 'TE_TE_TD_hot',
        'TE_TE_REC_hot_cluster', 'TE_TE_RECY_hot_cluster', 'TE_TE_TD_hot_cluster'
    ],
    'K': [
        'K_K_ATTEMPT_hot', 'K_K_FPTS_hot',
        'K_K_ATTEMPT_hot_cluster', 'K_K_FPTS_hot_cluster'
    ],
}

# metrics from the original csvs to average per cluster (adjust to what you like)
metrics_map = {
    'QB': ['FPTS','Yd','TD','Int','Rush_Yd'],
    'RB': ['FPTS','Rush_Yds','Tgt','Rec','Rec_Yds'],
    'WR': ['FPTS','Tgt','Rec','Rec_Yds'],
    'TE': ['FPTS','Tgt','Rec','Rec_Yds','Rec_TD'],
    'K':  ['FPTS','FGA','XP','XPAtt'],
}

# ---------- Build summaries ----------
qb_cluster_trends = summarize_pos_clusters('QB', prop_flags, qb_df, cluster_col_map, flag_cols_map, metrics_map)
rb_cluster_trends = summarize_pos_clusters('RB', prop_flags, rb_df, cluster_col_map, flag_cols_map, metrics_map)
wr_cluster_trends = summarize_pos_clusters('WR', prop_flags, wr_df, cluster_col_map, flag_cols_map, metrics_map)
te_cluster_trends = summarize_pos_clusters('TE', prop_flags, te_df, cluster_col_map, flag_cols_map, metrics_map)
k_cluster_trends  = summarize_pos_clusters('K',  prop_flags, k_df,  cluster_col_map, flag_cols_map, metrics_map)

# peek
for name, df in [
    ('QB clusters', qb_cluster_trends),
    ('RB clusters', rb_cluster_trends),
    ('WR clusters', wr_cluster_trends),
    ('TE clusters', te_cluster_trends),
    ('K clusters',  k_cluster_trends),
]:
    print(f'\n=== {name} ===')
    display(df.round(3))



=== QB clusters ===


,QB_cluster,n_teams,dist_mean,QB_QB_PASSY_hot,QB_QB_PTD_hot,QB_QB_RUSH_hot,QB_QB_PASSY_hot_cluster,QB_QB_PTD_hot_cluster,QB_QB_RUSH_hot_cluster,FPTS
0,0,21,1.901,0.238,0.190,0.095,0.0,0.0,0.0,18.010
1,2,6,1.807,0.500,0.667,0.833,1.0,1.0,1.0,25.967
2,1,5,1.708,0.000,0.000,0.600,0.0,0.0,1.0,17.126



=== RB clusters ===


,RB_cluster,n_teams,dist_mean,RB_RB_RUSH_hot,RB_RB_REC_hot,RB_RB_RECY_hot,RB_RB_RUSH_hot_cluster,RB_RB_REC_hot_cluster,RB_RB_RECY_hot_cluster,FPTS,Rush_Yds,Tgt,Rec,Rec_Yds
0,0,17,2.189,0.176,0.471,0.353,0.0,1.0,0.0,14.906,81.735,6.274,4.965,38.309
1,1,8,1.878,0.125,0.000,0.000,0.0,0.0,0.0,14.062,81.762,4.981,3.681,18.969
2,2,7,2.419,0.857,0.286,0.286,1.0,0.0,0.0,22.600,119.387,5.669,4.754,37.427



=== WR clusters ===


,WR_cluster,n_teams,dist_mean,WR_WR_REC_hot,WR_WR_RECY_hot,WR_WR_REC_hot_cluster,WR_WR_RECY_hot_cluster,FPTS,Tgt,Rec,Rec_Yds
0,1,17,1.558,0.059,0.000,0.0,0.0,17.459,17.253,10.876,130.535
1,2,12,1.386,0.750,0.667,1.0,1.0,22.319,20.542,13.460,165.173
2,0,3,1.240,0.000,0.333,0.0,0.0,22.800,16.467,10.000,152.433



=== TE clusters ===


,TE_cluster,n_teams,dist_mean,TE_TE_REC_hot,TE_TE_RECY_hot,TE_TE_TD_hot,TE_TE_REC_hot_cluster,TE_TE_RECY_hot_cluster,TE_TE_TD_hot_cluster,FPTS,Tgt,Rec,Rec_Yds,Rec_TD
0,2,13,1.352,0.538,0.154,0.538,1.0,0.0,0.0,8.223,8.152,6.125,53.125,0.560
1,1,12,1.583,0.000,0.000,0.167,0.0,0.0,0.0,4.442,5.623,3.758,34.738,0.248
2,0,7,1.350,0.571,0.857,0.429,0.0,1.0,0.0,9.329,7.379,5.893,69.500,0.450



=== K clusters ===


,K_cluster,n_teams,dist_mean,K_K_ATTEMPT_hot,K_K_FPTS_hot,K_K_ATTEMPT_hot_cluster,K_K_FPTS_hot_cluster,FPTS,FGA,XP,XPAtt
0,1,10,0.918,0.000,0.000,0.0,0.0,6.110,1.670,2.105,2.165
1,6,7,0.567,0.714,0.143,0.0,0.0,7.436,1.829,2.936,3.021
2,0,6,0.643,0.833,0.833,0.0,1.0,8.833,2.692,1.383,1.492
3,5,5,0.570,0.200,0.400,0.0,0.0,8.288,2.434,2.188,2.332
4,2,2,0.575,1.000,1.000,1.0,1.0,10.600,2.700,3.400,3.500
5,3,1,0.000,1.000,0.000,1.0,0.0,6.800,1.200,3.800,4.000
6,4,1,0.000,0.000,0.000,0.0,0.0,2.600,1.400,1.400,1.400


In [12]:
def pick_numeric(df, drop_cols=('Rank',)):
    use = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    return use.select_dtypes('number')

# --- QB ---
qb_X = pick_numeric(qb_df)
qb_scaler = StandardScaler().fit(qb_X)
qb_Z = qb_scaler.transform(qb_X)

best_k, best_s = None, -1
for k in range(3, 8):
    km = KMeans(n_clusters=k, n_init='auto', random_state=42)
    lbl = km.fit_predict(qb_Z)
    s = silhouette_score(qb_Z, lbl)
    if s > best_s:
        best_k, best_s, qb_km, qb_labels = k, s, km, lbl

qb_centroid_dist = qb_km.transform(qb_Z).min(axis=1)
qb_tags = pd.DataFrame({
    'Team': qb_df['Team'],
    'QB_cluster': qb_labels,
    'QB_dist': qb_centroid_dist,
    'QB_FPTS': qb_df['FPTS'] if 'FPTS' in qb_df.columns else None
})

# --- RB ---
rb_X = pick_numeric(rb_df)
rb_scaler = StandardScaler().fit(rb_X)
rb_Z = rb_scaler.transform(rb_X)

best_k, best_s = None, -1
for k in range(3, 8):
    km = KMeans(n_clusters=k, n_init='auto', random_state=42)
    lbl = km.fit_predict(rb_Z)
    s = silhouette_score(rb_Z, lbl)
    if s > best_s:
        best_k, best_s, rb_km, rb_labels = k, s, km, lbl

rb_centroid_dist = rb_km.transform(rb_Z).min(axis=1)
rb_tags = pd.DataFrame({
    'Team': rb_df['Team'],
    'RB_cluster': rb_labels,
    'RB_dist': rb_centroid_dist,
    'RB_FPTS': rb_df['FPTS'] if 'FPTS' in rb_df.columns else None
})

# --- WR ---
wr_X = pick_numeric(wr_df)
wr_scaler = StandardScaler().fit(wr_X)
wr_Z = wr_scaler.transform(wr_X)

best_k, best_s = None, -1
for k in range(3, 8):
    km = KMeans(n_clusters=k, n_init='auto', random_state=42)
    lbl = km.fit_predict(wr_Z)
    s = silhouette_score(wr_Z, lbl)
    if s > best_s:
        best_k, best_s, wr_km, wr_labels = k, s, km, lbl

wr_centroid_dist = wr_km.transform(wr_Z).min(axis=1)
wr_tags = pd.DataFrame({
    'Team': wr_df['Team'],
    'WR_cluster': wr_labels,
    'WR_dist': wr_centroid_dist,
    'WR_FPTS': wr_df['FPTS'] if 'FPTS' in wr_df.columns else None
})

# --- TE ---
te_X = pick_numeric(te_df)
te_scaler = StandardScaler().fit(te_X)
te_Z = te_scaler.transform(te_X)

best_k, best_s = None, -1
for k in range(3, 8):
    km = KMeans(n_clusters=k, n_init='auto', random_state=42)
    lbl = km.fit_predict(te_Z)
    s = silhouette_score(te_Z, lbl)
    if s > best_s:
        best_k, best_s, te_km, te_labels = k, s, km, lbl

te_centroid_dist = te_km.transform(te_Z).min(axis=1)
te_tags = pd.DataFrame({
    'Team': te_df['Team'],
    'TE_cluster': te_labels,
    'TE_dist': te_centroid_dist,
    'TE_FPTS': te_df['FPTS'] if 'FPTS' in te_df.columns else None
})

# --- K ---
k_X = pick_numeric(k_df)
k_scaler = StandardScaler().fit(k_X)
k_Z = k_scaler.transform(k_X)

best_k, best_s = None, -1
for k in range(3, 8):
    km = KMeans(n_clusters=k, n_init='auto', random_state=42)
    lbl = km.fit_predict(k_Z)
    s = silhouette_score(k_Z, lbl)
    if s > best_s:
        best_k, best_s, k_km, k_labels = k, s, km, lbl

k_centroid_dist = k_km.transform(k_Z).min(axis=1)
k_tags = pd.DataFrame({
    'Team': k_df['Team'],
    'K_cluster': k_labels,
    'K_dist': k_centroid_dist,
    'K_FPTS': k_df['FPTS'] if 'FPTS' in k_df.columns else None
})


In [17]:
meta = qb_tags.merge(rb_tags, on='Team') \
              .merge(wr_tags, on='Team') \
              .merge(te_tags, on='Team') \
              .merge(k_tags,  on='Team')

# Optional: just to check
meta.head(32)


,Team,QB_cluster,QB_dist,QB_FPTS,RB_cluster,RB_dist,RB_FPTS,WR_cluster,WR_dist,WR_FPTS,TE_cluster,TE_dist,TE_FPTS,K_cluster,K_dist,K_FPTS
0,Cowboys,2,2.602753,30.80,0,2.944611,19.60,2,1.929234,30.40,0,1.550689,6.6,3,1.151231e+00,8.40
1,Ravens,2,2.681598,28.20,4,1.429070,24.60,2,1.929234,28.00,0,1.517800,8.2,2,1.164280e+00,10.20
2,Dolphins,2,3.113186,26.80,5,2.860399,19.40,3,2.989317,17.80,2,2.001295,9.6,5,1.814975e+00,9.00
3,Steelers,2,2.440647,24.00,5,2.357803,16.00,1,3.113164,21.50,2,1.881731,10.5,1,5.847757e-01,6.00
4,Buccaneers,2,1.803746,23.60,6,1.202788,15.80,1,1.731064,19.00,0,2.123583,8.8,1,1.247453e+00,7.20
5,Jets,0,4.064964,23.20,1,2.212698,18.20,0,1.934331,21.00,0,2.933647,10.0,2,1.164280e+00,11.00
6,Bears,0,4.644084,23.00,2,2.175670,25.00,0,1.934331,23.00,0,2.100156,6.5,6,6.025682e-01,7.25
7,Saints,2,2.210027,22.40,5,1.807411,17.20,1,2.434459,17.60,0,1.604831,9.8,6,1.440042e+00,7.80
8,Bengals,0,2.832248,22.00,2,2.175670,26.80,5,1.486888,21.00,2,2.638906,10.8,1,3.054977e+00,6.80
9,Commanders,0,2.629884,21.20,6,2.276710,13.60,3,1.873811,22.80,0,3.492527,8.8,1,1.560740e+00,6.60


In [15]:
fpts_cols = [c for c in meta.columns if c.endswith('_FPTS')]
X_meta_fpts = meta[fpts_cols].copy()

sc = StandardScaler().fit(X_meta_fpts)
Z_meta_fpts = sc.transform(X_meta_fpts)

best_k, best_s = None, -1
for k in range(3, 8):
    km = KMeans(n_clusters=k, n_init='auto', random_state=42)
    lbl = km.fit_predict(Z_meta_fpts)
    s = silhouette_score(Z_meta_fpts, lbl)
    if s > best_s:
        best_k, best_s, meta_km, meta_labels = k, s, km, lbl

meta['DEF_archetype'] = meta_labels
meta[['Team','DEF_archetype'] + fpts_cols].sort_values('DEF_archetype').head(32)


,Team,DEF_archetype,QB_FPTS,RB_FPTS,WR_FPTS,TE_FPTS,K_FPTS
21,Colts,0,17.20,13.20,23.20,5.6,5.40
18,49ers,0,17.80,15.00,17.20,7.6,6.00
12,Jaguars,0,19.60,14.80,19.80,7.0,5.20
11,Patriots,0,19.80,12.60,20.80,9.4,5.80
10,Lions,0,20.20,12.00,24.40,7.6,5.60
9,Commanders,0,21.20,13.60,22.80,8.8,6.60
17,Packers,0,18.00,11.00,19.50,9.0,6.50
25,Browns,0,17.00,11.00,21.40,6.6,6.60
4,Buccaneers,0,23.60,15.80,19.00,8.8,7.20
3,Steelers,0,24.00,16.00,21.50,10.5,6.00


In [16]:
# one-hot encode per-position cluster labels
onehot = pd.get_dummies(meta[['QB_cluster','RB_cluster','WR_cluster','TE_cluster','K_cluster']].astype('category'),
                        prefix=['QB','RB','WR','TE','K'])

X_meta_rich = pd.concat([
    onehot,
    meta[['QB_dist','RB_dist','WR_dist','TE_dist','K_dist']].copy(),
    meta[[c for c in meta.columns if c.endswith('_FPTS')]].copy()
], axis=1)

sc = StandardScaler().fit(X_meta_rich)
Z = sc.transform(X_meta_rich)

best_k, best_s = None, -1
for k in range(3, 9):
    km = KMeans(n_clusters=k, n_init='auto', random_state=42)
    lbl = km.fit_predict(Z)
    s = silhouette_score(Z, lbl)
    if s > best_s:
        best_k, best_s, meta_km_rich, meta_labels_rich = k, s, km, lbl

meta['DEF_archetype_rich'] = meta_labels_rich
meta[['Team','DEF_archetype_rich']].head()


,Team,DEF_archetype_rich
0,Cowboys,2
1,Ravens,2
2,Dolphins,0
3,Steelers,0
4,Buccaneers,0
